# ResearchField AI — Balanced Abstract Classification

This notebook reruns the complete data-preparation and six-model experiment from the supplied project archive. It uses the three assignment classes: **Science (including Engineering)**, **Business**, and **Social Sciences**. The archive contains the original CSVs and readable source modules.

**Important:** model selection is based on validation macro F1. The held-out test set is used only for final evaluation. Labels are provisional labels inherited from the supplied CSV filenames.

In [ ]:
%pip -q install pandas numpy scikit-learn nltk joblib matplotlib seaborn pytest

## 1. Upload the project ZIP

Download `researchfield_ai_v2.zip` from the project deliverables, then upload it in the cell below. The ZIP includes the raw source CSVs and code. This avoids having to upload the six CSVs separately.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, shutil

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise ValueError('Please upload the researchfield_ai_v2.zip project archive.')
archive = zip_names[0]
extract_root = Path('/content/researchfield_ai_v2')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as zf:
    zf.extractall('/content')
# Support archives that either include a top-level folder or its contents directly.
if not (extract_root / 'src').exists():
    candidates = [p for p in Path('/content').iterdir() if p.is_dir() and (p/'src').exists() and (p/'data').exists()]
    if not candidates:
        raise FileNotFoundError('Could not locate extracted project folder containing src/ and data/.')
    extract_root = candidates[0]
os.chdir(extract_root)
print('Project directory:', Path.cwd())
print('Raw CSV files:', sorted(p.name for p in (Path('data/raw')).glob('*.csv')))

## 2. Rebuild the clean and balanced datasets

The preparation step audits cross-label EID conflicts, removes unusable abstracts and selected non-research document types, deduplicates records, and downsamples without replacement to equal class counts.

In [ ]:
from src.data_prep import prepare_dataset
cleaned, balanced, quality_report, class_distribution = prepare_dataset('data/raw', 'data/processed')
print('Cleaned unique records:', len(cleaned))
display(class_distribution)
print('Balanced records:', len(balanced))
display(balanced['label'].value_counts().rename_axis('label').reset_index(name='count'))
display(quality_report)

## 3. Train and compare six classifiers

NLTK tokenization, stopword filtering and Snowball stemming are part of the saved pipeline. TF-IDF is fitted only on the training split. Random seed is fixed for reproducibility.

In [ ]:
from src.train import evaluate
import pandas as pd, json
metadata, comparison = evaluate()
print(json.dumps(metadata, indent=2))
display(comparison)

## 4. Inspect the measured results

The numbers below are generated from this run, not hard-coded. The selected model is chosen using validation macro F1; the held-out test set is used for final evaluation.

In [ ]:
from IPython.display import display, Image
comparison = pd.read_csv('results/model_comparison.csv')
report = pd.read_csv('results/classification_report.csv', index_col=0)
print('Model comparison')
display(comparison)
print('Selected model test classification report')
display(report)
for image_path in ['results/class_distribution.png','results/validation_macro_f1.png','results/confusion_matrix.png']:
    if Path(image_path).exists():
        print(image_path)
        display(Image(filename=image_path))

## 5. Download the updated project

This cell packages the current files so you can download the regenerated model, datasets, results, and code.

In [ ]:
from google.colab import files
import shutil
output_zip = shutil.make_archive('/content/researchfield_ai_v2_colab_run', 'zip', root_dir=str(Path.cwd()))
print('Created:', output_zip)
files.download(output_zip)